# Desafio Módulo 4 - Sistema Ativo de Monitoramento e Rastreamento de Alvos

### [Conheça o projeto]

O objetivo deste desafio é construir o pipeline base para um sistema de segurança eletrônica ou monitoramento autônomo. Em aplicações de visão computacional em tempo real, rodar redes neurais profundas de detecção a cada milissegundo gera um custo computacional proibitivo. Para resolver isso, você irá implementar uma arquitetura híbrida inteligente: o sistema utiliza uma rede de detecção para localizar uma pessoa na cena (inicialização do alvo) e, nos quadros seguintes, transfere a responsabilidade para um algoritmo de *Tracking* (Rastreamento) matemático, muito mais leve e veloz.

> 💡 **Observação Didática:** Para este projeto, utilizaremos um vídeo curto de teste (`monitoramento_pessoas.mp4`) focado em cenários de vigilância. Isso permitirá validar a transição automática entre o estado de "Procura" (Detecção) e o estado de "Seguimento" (Rastreamento) sem sobrecarregar o hardware.
> 

### [Como carregar os materiais]

Você deve utilizar o arquivo de vídeo `monitoramento_pessoas.mp4` disponibilizado na plataforma do curso. Certifique-se de salvá-lo no mesmo diretório de execução do seu script ou notebook para que o OpenCV consiga ler o fluxo de mídia corretamente.

```python
import cv2

# Caminho para o vídeo do desafio
video_path = 'monitoramento_pessoas.mp4'

# Instanciação do fluxo de captura
video = cv2.VideoCapture(video_path)

if not video.isOpened():
    print(f"X Erro: Não foi possível abrir o arquivo {video_path}.")
else:
    print(f"✓ Vídeo carregado com sucesso!")
    video.release()
```

### [Instruções]

Neste desafio, você combinará o OpenCV com o modelo YOLO para criar um fluxo automatizado. Diferente do rastreamento puramente manual (onde o usuário clica e arrasta para selecionar a pessoa), seu código deverá ler o primeiro *frame*, usar o *head* de detecção da YOLO para encontrar a primeira pessoa disponível com alta confiança, extrair a caixa delimitadora (*bounding box*) e alimentar nativamente o inicializador de um rastreador do OpenCV (como o `TrackerCSRT`).

### [Exemplo de Implementação]

Veja abaixo como extrair as coordenadas de uma detecção da YOLO e convertê-las para o formato aceito pelos rastreadores do OpenCV `(x, y, largura, altura)`:

```python
from ultralytics import YOLO
import cv2

# --- ESTADO 1: DETECÇÃO AUTOMÁTICA (Frame Inicial) ---
ok, frame_inicial = cap.read()
if ok:
    model = YOLO(YOLO_MODEL_NAME)
    results = model.predict(source=frame_inicial, classes=[0], conf=CONFIDENCE_THRESHOLD, verbose=False)
    
    if len(results[0].boxes) > 0:
        box = results[0].boxes[0].xywh[0].tolist()
        bbox_opencv = (int(box[0] - box[2]/2), int(box[1] - box[3]/2), int(box[2]), int(box[3]))
        
        # --- ESTADO 2: TRANSIÇÃO ---
        tracker = criar_tracker(TIPO_TRACKER)
        tracker.init(frame_inicial, bbox_opencv)
        alvo_detectado = True
```

### [Requisitos Técnicos]

- **Gatilhamento Automático (Detecção):** O pipeline não deve depender de seleção manual (`cv2.selectROI`). O sistema precisa analisar o primeiro frame do vídeo usando YOLO e selecionar automaticamente a pessoa com maior nível de confiança para ser o alvo.
- **Transição de Estados:** Assim que o alvo for detectado, a rede YOLO deve ser "desligada" (encerrada) e o algoritmo `TrackerCSRT` (ou `TrackerKCF`) do OpenCV deve ser inicializado com as coordenadas obtidas.
- **Métricas em Tempo Real:** O loop principal deve processar o restante do vídeo apenas via *tracker*. O sistema deve exibir na tela se o estado atual é de "Rastreando Alvo" ou se o alvo foi perdido.
- **Tratamento de Perda de Alvo:** Se o objeto sofrer uma oclusão total ou sair do vídeo, o *tracker* retornará um status negativo. O código deve capturar essa exceção de forma limpa, exibindo um alerta visual em vermelho na tela (`"ALVO PERDIDO"`).

### [Tarefas]

- [ ]  Implementar a detecção automática da pessoa no frame número 1 do vídeo utilizando o modelo YOLO pré-treinado.
- [ ]  Converter com precisão matemática as coordenadas retornadas pela estrutura da Ultralytics para o formato de matriz do OpenCV.
- [ ]  Instanciar um algoritmo de rastreamento do OpenCV (dando preferência ao **CSRT** pela variação de escala e postura do corpo humano) e vinculá-lo ao alvo detectado.
- [ ]  Criar o laço (*loop*) de repetição para percorrer o restante do vídeo frame a frame atualizando a posição da *bounding box* sem chamar a rede YOLO novamente.
- [ ]  Adicionar elementos visuais na exibição: uma caixa verde ao redor da pessoa rastreada, um texto indicando a precisão/sucesso do rastreio e um contador de frames no canto superior esquerdo da tela.

### [Diretrizes para o Código]

- **Eficiência Computacional:** Como a detecção ocorre apenas uma vez e os frames seguintes usam apenas cálculos matemáticos de correlação espacial do *tracker*, o vídeo deve rodar de forma extremamente fluida (com alto FPS).
- **Robustez:** Teste o comportamento do rastreador caso a pessoa passe por trás de algum obstáculo no vídeo e observe as limitações físicas do modelo visual baseado em aparência.

Feito com 💜 por Rocketseat 👋

In [36]:
%pip install ultralytics
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [37]:
import cv2
import time
from ultralytics import YOLO

print("✓ Dependências importadas com sucesso!")
print(f"OpenCV Versão: {cv2.__version__}")

✓ Dependências importadas com sucesso!
OpenCV Versão: 4.13.0


In [38]:
# Configurações do projeto
VIDEO_PATH = 'Videos/monitoramento_pessoas3.mp4'
YOLO_MODEL_NAME = 'yolov8n.pt'
CONFIDENCE_THRESHOLD = 0.5
CLASS_PERSON_ID = 0  # Classe correspondente a 'person' no dataset COCO

# Testando se o OpenCV consegue ler o vídeo
cap_teste = cv2.VideoCapture(VIDEO_PATH)
if not cap_teste.isOpened():
    print(f"X Erro: Não foi possível encontrar ou abrir o arquivo '{VIDEO_PATH}'.")
else:
    print(f"✓ Arquivo de vídeo validado com sucesso!")
    cap_teste.release()

    # --- CAPTURA DO FPS ORIGINAL DO VÍDEO ---
    ORIGINAL_FPS = cap_teste.get(cv2.CAP_PROP_FPS)
    # Tratamento de exceção caso o metadado venha zerado ou inválido
    if ORIGINAL_FPS <= 0:
        ORIGINAL_FPS = 24.0  # Fallback padrão de cinema

    FRAME_TIME = 1.0 / ORIGINAL_FPS
    print(f"✓ Taxa de quadros detectada no vídeo original: {ORIGINAL_FPS:.2f} FPS")
    print(f"✓ Tempo de exibição calculado por frame: {FRAME_TIME * 1000:.2f}ms")

TIPO_TRACKER = "MOSSE"

✓ Arquivo de vídeo validado com sucesso!
✓ Taxa de quadros detectada no vídeo original: 24.00 FPS
✓ Tempo de exibição calculado por frame: 41.67ms


In [39]:
# Criar Tracker

def criar_tracker(tipo):
    tipo = tipo.upper()
    if tipo == "MOSSE":
        return cv2.legacy.TrackerMOSSE_create()
    elif tipo == "CSRT":
        return cv2.TrackerCSRT_create()
    elif tipo == "KCF":
        return cv2.TrackerKCF_create()
    elif tipo == "MIL":
        return cv2.TrackerMIL_create()
    elif tipo == "BOOSTING":
        return cv2.legacy.TrackerBoosting_create()

In [ ]:
# 1. Inicialização e Captura de Metadados
cap = cv2.VideoCapture(VIDEO_PATH)

# Variáveis de controle
prev_time = time.time()
alvo_detectado = False
tracker = None

# --- ESTADO 1: DETECÇÃO AUTOMÁTICA (Frame Inicial) ---
ok, frame_inicial = cap.read()
if ok:
    model = YOLO(YOLO_MODEL_NAME)
    results = model.predict(source=frame_inicial, classes=[0], conf=CONFIDENCE_THRESHOLD, verbose=False)
    
    if len(results[0].boxes) > 0:
        box = results[0].boxes[0].xywh[0].tolist()
        bbox_opencv = (int(box[0] - box[2]/2), int(box[1] - box[3]/2), int(box[2]), int(box[3]))
        
        # --- ESTADO 2: TRANSIÇÃO ---
        tracker = criar_tracker(TIPO_TRACKER)
        tracker.init(frame_inicial, bbox_opencv)
        alvo_detectado = True

# --- ESTADO 2: MONITORAMENTO ---
if alvo_detectado:
    while True:
        loop_start = time.time()
        ok, frame = cap.read()
        if not ok: break
            
        # 1. Cálculo de FPS Real (Processamento)
        current_time = time.time()
        fps_real = 1 / (current_time - prev_time)
        prev_time = current_time
        
        # 2. Cálculo do Tempo do Vídeo (Duração Momento)
        msec = cap.get(cv2.CAP_PROP_POS_MSEC)
        segundos_totais = int(msec / 1000)
        minutos = segundos_totais // 60
        segundos = segundos_totais % 60
        timestamp = f"{minutos:02d}:{segundos:02d}"
        
        # 3. Atualização do Rastreamento
        sucesso, bbox = tracker.update(frame)
        
        # --- INTERFACE VISUAL (UI) ---
        # Tarja de Telemetria (Fundo semi-transparente)
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (400, 130), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        # Textos de Telemetria
        cv2.putText(frame, f"MODELO: {TIPO_TRACKER}", (20, 35), 2, 0.6, (0, 255, 255), 2)
        cv2.putText(frame, f"FPS REAL: {int(fps_real)}", (20, 70), 2, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f"TEMPO: {timestamp}", (20, 105), 2, 0.6, (200, 255, 200), 2)
        
        if sucesso:
            (x, y, w, h) = [int(v) for v in bbox]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, "RASTREANDO", (x, y - 10), 0, 0.5, (0, 255, 0), 1)
        else:
            cv2.putText(frame, "ALVO PERDIDO", (150, 200), 0, 1, (0, 0, 255), 3)

        cv2.imshow("Monitoramento de Alta Performance", frame)
        
        # --- COMPENSAÇÃO DE VELOCIDADE (FPS ORIGINAL) ---
        elapsed = time.time() - loop_start
        wait = FRAME_TIME - elapsed
        if wait > 0: time.sleep(wait)
        
        if cv2.waitKey(1) & 0xFF in [27, ord('q')]: break

cap.release()
cv2.destroyAllWindows()